# 01 Switching Penalty Sweep

This notebook visualizes the **switching penalty sweep** experiment.

- Input:
  - `result/table/01_switch_penalty_summary.csv`
  - `result/table/01_switch_penalty_timeseries.csv`
- Outputs:
  - PDF vector figures in `result/figure/`
  - summary tables (mean and standard deviation) in `result/table/`


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# -----------------------------------------------------------------------------
# Fix: Handle paths for both Script (.py) and Jupyter Notebook (.ipynb)
# -----------------------------------------------------------------------------
try:
    # This works when running as a script
    current_path = Path(__file__).resolve()
    ROOT = current_path.parents[1]
except NameError:
    # This works in Jupyter Notebook
    current_path = Path.cwd()
    ROOT = current_path.parents[0]

# Verify the path is correct
print(f"Current ROOT: {ROOT}")

tbl_dir = ROOT / "result" / "table"
fig_dir = ROOT / "result" / "figure"
fig_dir.mkdir(parents=True, exist_ok=True)

# -----------------------------------------------------------------------------
# Load Data
# -----------------------------------------------------------------------------
summary_csv = tbl_dir / "01_switch_penalty_summary.csv"
timeseries_csv = tbl_dir / "01_switch_penalty_timeseries.csv"

if summary_csv.exists() and timeseries_csv.exists():
    df_sum = pd.read_csv(summary_csv)
    df_ts = pd.read_csv(timeseries_csv)
    print("Summary df head:")
    print(df_sum.head())
else:
    if not summary_csv.exists():
        print(f"Error: File not found at {summary_csv}")
    if not timeseries_csv.exists():
        print(f"Error: File not found at {timeseries_csv}")


In [ ]:
# Aggregate mean/std across runs
metrics = ["avg_sum_queue", "p95_max_queue", "avg_sense_u", "p95_sense_u",
           "alpha_total_variation", "switch_count", "deadline_violation_rate", "avg_slot_ms"]

agg = df_sum.groupby("switch_penalty")[metrics].agg(["mean", "std"]).reset_index()
agg.columns = ["switch_penalty"] + [f"{m}_{s}" for m in metrics for s in ["mean", "std"]]

out_table = tbl_dir / "01_switch_penalty_table.csv"
agg.to_csv(out_table, index=False)
print("Saved table:", out_table)

agg


In [ ]:
def errorbar_plot(x, y_mean, y_std, xlabel, ylabel, fname):
    plt.figure()
    plt.errorbar(x, y_mean, yerr=y_std, fmt='o-', capsize=3)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.grid(True, alpha=0.3)
    out = fig_dir / fname
    plt.savefig(out, format="pdf", bbox_inches="tight")
    print("Saved:", out)

x = agg["switch_penalty"].values

errorbar_plot(
    x,
    agg["avg_sum_queue_mean"].values,
    agg["avg_sum_queue_std"].values,
    xlabel="Switching penalty $\lambda_{\mathrm{sw}}$",
    ylabel="Average total queue",
    fname="01_switch_penalty_avg_queue.pdf",
)

errorbar_plot(
    x,
    agg["avg_sense_u_mean"].values,
    agg["avg_sense_u_std"].values,
    xlabel="Switching penalty $\lambda_{\mathrm{sw}}$",
    ylabel="Average sensing uncertainty $u_t$",
    fname="01_switch_penalty_avg_sense_u.pdf",
)


In [ ]:
# Gate overhead: total variation and switch counts
plt.figure()
plt.errorbar(
    x,
    agg["alpha_total_variation_mean"].values,
    yerr=agg["alpha_total_variation_std"].values,
    fmt="o-",
    capsize=3,
)
plt.xlabel("Switching penalty $\lambda_{\mathrm{sw}}$")
plt.ylabel("Total variation of $\alpha_t$")
plt.grid(True, alpha=0.3)
out = fig_dir / "01_switch_penalty_alpha_tv.pdf"
plt.savefig(out, format="pdf", bbox_inches="tight")
print("Saved:", out)

plt.figure()
plt.errorbar(
    x,
    agg["switch_count_mean"].values,
    yerr=agg["switch_count_std"].values,
    fmt="o-",
    capsize=3,
)
plt.xlabel("Switching penalty $\lambda_{\mathrm{sw}}$")
plt.ylabel("Gate switch count")
plt.grid(True, alpha=0.3)
out = fig_dir / "01_switch_penalty_switch_count.pdf"
plt.savefig(out, format="pdf", bbox_inches="tight")
print("Saved:", out)


In [ ]:
# Pareto scatter: (avg_sum_queue, avg_sense_u) for each lambda
pareto = df_sum.groupby(["switch_penalty"], as_index=False).agg(
    avg_sum_queue=("avg_sum_queue", "mean"),
    avg_sense_u=("avg_sense_u", "mean"),
)

plt.figure()
plt.scatter(pareto["avg_sum_queue"], pareto["avg_sense_u"])
for _, r in pareto.iterrows():
    plt.annotate(f'{r["switch_penalty"]:.2f}', (r["avg_sum_queue"], r["avg_sense_u"]))
plt.xlabel("Average total queue")
plt.ylabel("Average sensing uncertainty $u_t$")
plt.grid(True, alpha=0.3)
out = fig_dir / "01_switch_penalty_pareto.pdf"
plt.savefig(out, format="pdf", bbox_inches="tight")
print("Saved:", out)


In [ ]:
# Show how alpha(t) changes for a low vs high switching penalty.
# We average alpha across runs for stability.
low_lam = float(np.min(df_ts["switch_penalty"]))
high_lam = float(np.max(df_ts["switch_penalty"]))

g = df_ts.groupby(["switch_penalty", "t"], as_index=False).agg(alpha=("alpha", "mean"))

plt.figure()
for lam in [low_lam, high_lam]:
    sub = g[g["switch_penalty"] == lam]
    plt.plot(sub["t"], sub["alpha"], label=f"$\lambda_{{sw}}={lam:.2f}$")
plt.xlabel("Slot $t$")
plt.ylabel("Gate $\alpha_t$")
plt.legend()
plt.grid(True, alpha=0.3)
out = fig_dir / "01_switch_penalty_alpha_timeseries.pdf"
plt.savefig(out, format="pdf", bbox_inches="tight")
print("Saved:", out)
